In [13]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [14]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [15]:
simulator = BasicSimulator()

def quantum_random_bit():
    """Return a single truly-random bit by preparing |+> and measuring it."""
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)
    result = simulator.run(transpile(qc, simulator), shots=1).result()
    return int(list(result.get_counts().keys())[0])

# Number of qubits to send.
N = 100

# Threshold for reporting an attack: if the disagreement rate on the sample
# of compared bits exceeds this fraction, the protocol declares Eve present.
# Without an attacker, the expected rate is 0. With intercept-resend it is 0.25.
# A threshold of 0.15 sits safely between those two cases.
THRESHOLD = 0.15


In [16]:
# ALICE code

alice_bits = []
alice_bases = []
for i in range(N):
    alice_bits.append(quantum_random_bit())
    alice_bases.append(quantum_random_bit())

print(f"Alice generated {N} bits and {N} bases.")

Alice generated 100 bits and 100 bases.


In [17]:
# prepare and send the qubits

qubits = []
for i in range(N):
    qc = QuantumCircuit(1, 1)
    if alice_bits[i] == 1:
        qc.x(0)
    if alice_bases[i] == 1:
        qc.h(0)
    qubits.append(qc)

print(f"Alice has prepared {len(qubits)} qubits and sent them down the channel.")

Alice has prepared 100 qubits and sent them down the channel.


In [18]:

# EVE (attacker)
eve_bases = []
eve_bits = []          # what Eve learns (her guess at Alice's bit)
forwarded_qubits = []  # what Bob actually receives

for i in range(N):
    e_basis = quantum_random_bit()
    eve_bases.append(e_basis)

    qc = qubits[i].copy()
    if e_basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    result = simulator.run(transpile(qc, simulator), shots=1).result()
    e_bit = int(list(result.get_counts().keys())[0])
    eve_bits.append(e_bit)

    new_qc = QuantumCircuit(1, 1)
    if e_bit == 1:
        new_qc.x(0)
    if e_basis == 1:
        new_qc.h(0)
    forwarded_qubits.append(new_qc)

print(f"Eve has measured and forwarded {len(forwarded_qubits)} qubits.")

Eve has measured and forwarded 100 qubits.


In [19]:
# BOB code

bob_bases = []
for i in range(N):
    bob_bases.append(quantum_random_bit())

bob_bits = []
for i in range(N):
    qc = forwarded_qubits[i].copy()
    if bob_bases[i] == 1:
        qc.h(0)
    qc.measure(0, 0)
    result = simulator.run(transpile(qc, simulator), shots=1).result()
    outcome = int(list(result.get_counts().keys())[0])
    bob_bits.append(outcome)

print(f"Bob has measured {N} qubits.")

Bob has measured 100 qubits.


In [20]:
# PUBLIC CLASSICAL CHANNEL (Alice <-> Bob)
alice_key = []
bob_key = []
for i in range(N):
    if alice_bases[i] == bob_bases[i]:
        alice_key.append(alice_bits[i])
        bob_key.append(bob_bits[i])

print(f"Sifted key length: {len(alice_key)} out of {N} qubits sent.")

Sifted key length: 47 out of 100 qubits sent.


In [21]:
# attack detection
sample_alice = []
sample_bob = []
remaining_alice = []
remaining_bob = []

for i in range(len(alice_key)):
    if quantum_random_bit() == 1:
        sample_alice.append(alice_key[i])
        sample_bob.append(bob_key[i])
    else:
        remaining_alice.append(alice_key[i])
        remaining_bob.append(bob_key[i])

disagreements = 0
for i in range(len(sample_alice)):
    if sample_alice[i] != sample_bob[i]:
        disagreements += 1

if len(sample_alice) == 0:
    error_rate = 0.0
else:
    error_rate = disagreements / len(sample_alice)

print(f"Sample size                  : {len(sample_alice)}")
print(f"Disagreements in sample      : {disagreements}")
print(f"Observed error rate          : {error_rate:.3f}")
print(f"Threshold                    : {THRESHOLD}")
print(f"Expected rate with no attack : 0.000")
print(f"Expected rate with eavesdrop : 0.250")

Sample size                  : 23
Disagreements in sample      : 7
Observed error rate          : 0.304
Threshold                    : 0.15
Expected rate with no attack : 0.000
Expected rate with eavesdrop : 0.250


In [22]:
# RESULT
if error_rate > THRESHOLD:
    print("*** ATTACK DETECTED ***")
    print(f"Error rate {error_rate:.3f} exceeds threshold {THRESHOLD}.")
    print("Alice and Bob abort the protocol; the key is discarded.")
else:
    print("No attack detected.")
    print(f"Final shared key length: {len(remaining_alice)}")
    print(f"Final keys identical?  {remaining_alice == remaining_bob}")

*** ATTACK DETECTED ***
Error rate 0.304 exceeds threshold 0.15.
Alice and Bob abort the protocol; the key is discarded.


In [23]:
# REPEATED TRIALS
TRIALS = 10
TRIAL_N = 100

detections = 0
rates = []

for trial in range(TRIALS):
    # --- Alice ---
    a_bits = [quantum_random_bit() for _ in range(TRIAL_N)]
    a_bases = [quantum_random_bit() for _ in range(TRIAL_N)]
    a_qubits = []
    for i in range(TRIAL_N):
        qc = QuantumCircuit(1, 1)
        if a_bits[i] == 1:
            qc.x(0)
        if a_bases[i] == 1:
            qc.h(0)
        a_qubits.append(qc)

    # --- Eve (intercept-resend) ---
    fwd = []
    for i in range(TRIAL_N):
        eb = quantum_random_bit()
        qc = a_qubits[i].copy()
        if eb == 1:
            qc.h(0)
        qc.measure(0, 0)
        res = simulator.run(transpile(qc, simulator), shots=1).result()
        ev_bit = int(list(res.get_counts().keys())[0])
        new_qc = QuantumCircuit(1, 1)
        if ev_bit == 1:
            new_qc.x(0)
        if eb == 1:
            new_qc.h(0)
        fwd.append(new_qc)

    # --- Bob ---
    b_bases = [quantum_random_bit() for _ in range(TRIAL_N)]
    b_bits = []
    for i in range(TRIAL_N):
        qc = fwd[i].copy()
        if b_bases[i] == 1:
            qc.h(0)
        qc.measure(0, 0)
        res = simulator.run(transpile(qc, simulator), shots=1).result()
        b_bits.append(int(list(res.get_counts().keys())[0]))

    # --- Sift ---
    a_key = []
    b_key = []
    for i in range(TRIAL_N):
        if a_bases[i] == b_bases[i]:
            a_key.append(a_bits[i])
            b_key.append(b_bits[i])

    # --- Detection: sample everything for a tight estimate of the error rate ---
    if len(a_key) == 0:
        rate = 0.0
    else:
        diff = 0
        for i in range(len(a_key)):
            if a_key[i] != b_key[i]:
                diff += 1
        rate = diff / len(a_key)
    rates.append(rate)
    if rate > THRESHOLD:
        detections += 1
    print(f"Trial {trial + 1:>2}: sifted size = {len(a_key):>3}, error rate = {rate:.3f}, detected = {rate > THRESHOLD}")

avg = sum(rates) / TRIALS
print()
print(f"Average error rate across {TRIALS} trials: {avg:.3f}  (theoretical: 0.250)")
print(f"Attack detected in {detections} / {TRIALS} trials.")

Trial  1: sifted size =  45, error rate = 0.356, detected = True
Trial  2: sifted size =  48, error rate = 0.292, detected = True
Trial  3: sifted size =  44, error rate = 0.227, detected = True
Trial  4: sifted size =  46, error rate = 0.130, detected = False
Trial  5: sifted size =  49, error rate = 0.327, detected = True
Trial  6: sifted size =  51, error rate = 0.294, detected = True
Trial  7: sifted size =  58, error rate = 0.259, detected = True
Trial  8: sifted size =  48, error rate = 0.250, detected = True
Trial  9: sifted size =  49, error rate = 0.204, detected = True
Trial 10: sifted size =  48, error rate = 0.188, detected = True

Average error rate across 10 trials: 0.253  (theoretical: 0.250)
Attack detected in 9 / 10 trials.


In [24]:
#partial intercept resend
EXTRA_N = 200
K = 4                  # resolution: probabilities are multiples of 1/16
TOTAL = 2 ** K         # = 16

print(f"{'p':>6}  {'error rate':>11}  {'detected (>%.2f)':>17}" % THRESHOLD)
print("-" * 40)

for numerator in [0, 2, 4, 8, 12, 16]:
    p = numerator / TOTAL

    # --- Alice ---
    a_bits = [quantum_random_bit() for _ in range(EXTRA_N)]
    a_bases = [quantum_random_bit() for _ in range(EXTRA_N)]
    a_qubits = []
    for i in range(EXTRA_N):
        qc = QuantumCircuit(1, 1)
        if a_bits[i] == 1:
            qc.x(0)
        if a_bases[i] == 1:
            qc.h(0)
        a_qubits.append(qc)

    # --- Eve: attack with probability numerator/TOTAL ---
    fwd = []
    for i in range(EXTRA_N):
        # K quantum coins -> K-bit integer in [0, TOTAL).
        v = 0
        for _ in range(K):
            v = (v << 1) | quantum_random_bit()
        if v < numerator:
            # Eve attacks this qubit.
            eb = quantum_random_bit()
            qc = a_qubits[i].copy()
            if eb == 1:
                qc.h(0)
            qc.measure(0, 0)
            res = simulator.run(transpile(qc, simulator), shots=1).result()
            ev_bit = int(list(res.get_counts().keys())[0])
            new_qc = QuantumCircuit(1, 1)
            if ev_bit == 1:
                new_qc.x(0)
            if eb == 1:
                new_qc.h(0)
            fwd.append(new_qc)
        else:
            # Eve lets this qubit pass untouched.
            fwd.append(a_qubits[i])

    # --- Bob ---
    b_bases = [quantum_random_bit() for _ in range(EXTRA_N)]
    b_bits = []
    for i in range(EXTRA_N):
        qc = fwd[i].copy()
        if b_bases[i] == 1:
            qc.h(0)
        qc.measure(0, 0)
        res = simulator.run(transpile(qc, simulator), shots=1).result()
        b_bits.append(int(list(res.get_counts().keys())[0]))

    # --- Sift and measure error rate ---
    a_key, b_key = [], []
    for i in range(EXTRA_N):
        if a_bases[i] == b_bases[i]:
            a_key.append(a_bits[i])
            b_key.append(b_bits[i])
    if len(a_key) == 0:
        rate = 0.0
    else:
        diff = sum(1 for i in range(len(a_key)) if a_key[i] != b_key[i])
        rate = diff / len(a_key)
    print(f"{p:>6.2f}  {rate:>11.3f}  {str(rate > THRESHOLD):>17}")

     p   error rate   detected (>0.15)
----------------------------------------
  0.00        0.000              False
  0.12        0.022              False
  0.25        0.031              False
  0.50        0.116              False
  0.75        0.269               True
  1.00        0.290               True
